# Embedding Evaluation — Notebook

End-to-end walkthrough using `emb_tight.json` (tight scenario) and `emb_sparse.json` (sparse scenario). Run all cells top-to-bottom; all figures are interactive (Plotly).

**Tight** — orthogonal class prototypes, tiny per-sample noise. Intra-class cosine ≈ 0.99, inter-class ≈ 0.00. All KPIs near-perfect.

**Sparse** — nearby class prototypes (~60° apart), large per-sample noise. Classes overlap heavily — gap ≈ 0.07, purity@5 ≈ 0.56.

In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from pai.ag_emb.services.evaluate import run_evaluation
from pai.ag_emb.services.reporting import (
    plot_cosine_similarity,
    plot_knn_confusion,
    plot_lle,
    plot_tsne,
    print_result,
)

---

## Tight Scenario

### Load Data

In [2]:
payload_path = Path("emb_tight.json")
if not payload_path.exists():
    raise FileNotFoundError("emb_tight.json not found — start Jupyter from the examples/ directory")

with open(payload_path) as f:
    payload = json.load(f)

embeddings: dict[str, list[float]] = payload["embeddings"]
print(f"Loaded {len(embeddings)} embeddings  (dim={len(next(iter(embeddings.values())))})")

Loaded 23 embeddings  (dim=32)


### Run Evaluation

In [3]:
result = run_evaluation(
    image_embeddings=embeddings,
    k_values=[5, 10],
    dataset_root="images",
    sample_pairs=None,
)

print_result(result)

Confusion matrix: 100%|██████████| 8/8 [00:00<00:00, 74.10step/s]         

n_items      : 23
embedding_dim: 32
classes      : ['corn', 'soybean']
k_values     : [5, 10]

── global_metrics ──────────────────────────────────────────────────────
  pairwise cosine    : mean=0.5345  std=0.4725  (p05=-0.0237  p50=0.9187  p95=0.9907)
  centroid cosine    : mean=0.7448  std=0.2318  norm=0.7448
  intra/inter gap    : 0.9493  (intra=0.9548  inter=0.0055)
  effective_rank     : 1.22  (ratio=0.0380  dim=32)
  uniformity         : -0.7399

  hubness@5         : mean=5.0000  std=1.9111  p95=7.0000
  hubness@10        : mean=10.0000  std=4.7822  p95=20.8000
  knn_radius@5         : mean=0.9659  std=0.0330  p05=0.9090  p95=0.9900
  knn_radius@10        : mean=0.6538  std=0.4190  p05=0.0020  p95=0.9353
  mean_top_k_sim@5         : mean=0.9789  std=0.0170  p05=0.9452  p95=0.9913
  mean_top_k_sim@10        : mean=0.8518  std=0.1812  p05=0.5724  p95=0.9738
  outlier_score@5         : mean=0.0211  std=0.0170  p95=0.0548
  outlier_score@10        : mean=0.1482  std=0.1812  p95=0.4

### Visualizations

All plots are interactive — hover for details, click legend entries to toggle classes, and drag to rotate 3D views.

#### KNN Confusion Matrix

Rows = true class, columns = neighbor class, values = fraction of k-NN neighbors belonging to each class. The diagonal equals mean KNN purity — higher is better.

In [4]:
plot_knn_confusion(result, output_path=None)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'colorbar': {'thickness': 18, 'title': {'text': 'KNN@10<br>neighbor<br>fraction'}},
              'colorscale': [[0.0, 'rgb(165,0,38)'], [0.1, 'rgb(215,48,39)'],
                             [0.2, 'rgb(244,109,67)'], [0.3, 'rgb(253,174,97)'],
                             [0.4, 'rgb(254,224,139)'], [0.5, 'rgb(255,255,191)'],
                             [0.6, 'rgb(217,239,139)'], [0.7, 'rgb(166,217,106)'],
                             [0.8, 'rgb(102,189,99)'], [0.9, 'rgb(26,152,80)'],
                             [1.0, 'rgb(0,104,55)']],
              'hovertemplate': ('True class: <b>%{y}</b><br>Nei' ... 'ction: %{z:.4f}<extra></extra>'),
              'text': [['1.000', '0.000'], ['0.400', '0.600']],
              'textfont': {'size': 13},
              'texttemplate': '%{text}',
              'type': 'heatmap',
              'x': [corn, soybean],
              'y': [corn, soybean],
              'z': [[1.0, 0.0], [0.4, 0.6]],
              'zmax': 1.0,
              'zmin': 0.0}],
    'layout': {'height': 530,
               'margin': {'b': 100, 'l': 120, 'r': 60, 't': 70},
               'paper_bgcolor': 'white',
               'plot_bgcolor': 'white',
               'template': '...',
               'title': {'font': {'size': 17}, 'text': 'KNN Confusion Matrix  (k=10)'},
               'width': 560,
               'xaxis': {'side': 'bottom', 'tickfont': {'size': 12}, 'title': {'text': 'Neighbor class'}},
               'yaxis': {'autorange': 'reversed', 'tickfont': {'size': 12}, 'title': {'text': 'True class'}}}
})

#### Pairwise Cosine Similarity

Full N×N cosine similarity matrix sorted by class. Within-class blocks sit on the diagonal — tighter, brighter blocks indicate a more discriminative embedding space.

In [5]:
plot_cosine_similarity(embeddings, result, output_path=None)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'colorbar': {'thickness': 18, 'title': {'text': 'Cosine<br>similarity'}},
              'colorscale': [[0.0, 'rgb(5,48,97)'], [0.1, 'rgb(33,102,172)'],
                             [0.2, 'rgb(67,147,195)'], [0.3, 'rgb(146,197,222)'],
                             [0.4, 'rgb(209,229,240)'], [0.5, 'rgb(247,247,247)'],
                             [0.6, 'rgb(253,219,199)'], [0.7, 'rgb(244,165,130)'],
                             [0.8, 'rgb(214,96,77)'], [0.9, 'rgb(178,24,43)'],
                             [1.0, 'rgb(103,0,31)']],
              'hovertemplate': 'Row: %{y}<br>Col: %{x}<br>Cosine similarity: %{z:.4f}<extra></extra>',
              'type': 'heatmap',
              'x': [220622-225403-corn-HB-25000SBC-54e94639.png,
                    220622-225403-corn-HB-25000SBC-af33e7ca.png,
                    220622-225403-corn-HB-25000SBC-fda2adee.png,
                    220622-225404-corn-HB-25000SBC-58235c1f.png,
                    220622-225405-corn-HB-25000SBC-328a4c61.png,
                    220622-225405-corn-HB-25000SBC-68fd040c.png,
                    220622-225405-corn-HB-25000SBC-7a2bde6a.png,
                    220622-225405-corn-HB-25000SBC-a45a936e.png,
                    190627-090900-corn-nikon_d610-bded43cb.JPG, 190627-090901-corn-
                    nikon_d610-67679dac.JPG, 190627-091105-corn-
                    nikon_d610-8d6b86d0.JPG, 190627-091202-corn-
                    nikon_d610-b7725cb0.JPG, 190627-090803-corn-
                    nikon_d610-84e374af.JPG, 190627-091502-corn-
                    nikon_d610-ba4f7547.JPG, 190627-091503-corn-
                    nikon_d610-e2cc5642.JPG, 190627-091801-corn-
                    nikon_d610-94fd680b.JPG, 220608-143001-soybeans-
                    HB-25000SBC-37dc1b2c.png, 220608-143001-soybeans-
                    HB-25000SBC-754c6a11.png, 220608-143001-soybeans-
                    HB-25000SBC-8235f317.png, 210625-131202-soybeans-Anafi-
                    fce5a850.JPG, 210625-131205-soybeans-Anafi-8bc119e3.JPG,
                    210625-131303-soybeans-Anafi-614a4aea.JPG,
                    210625-131501-soybeans-Anafi-49b5fd6d.JPG],
              'y': [220622-225403-corn-HB-25000SBC-54e94639.png,
                    220622-225403-corn-HB-25000SBC-af33e7ca.png,
                    220622-225403-corn-HB-25000SBC-fda2adee.png,
                    220622-225404-corn-HB-25000SBC-58235c1f.png,
                    220622-225405-corn-HB-25000SBC-328a4c61.png,
                    220622-225405-corn-HB-25000SBC-68fd040c.png,
                    220622-225405-corn-HB-25000SBC-7a2bde6a.png,
                    220622-225405-corn-HB-25000SBC-a45a936e.png,
                    190627-090900-corn-nikon_d610-bded43cb.JPG, 190627-090901-corn-
                    nikon_d610-67679dac.JPG, 190627-091105-corn-
                    nikon_d610-8d6b86d0.JPG, 190627-091202-corn-
                    nikon_d610-b7725cb0.JPG, 190627-090803-corn-
                    nikon_d610-84e374af.JPG, 190627-091502-corn-
                    nikon_d610-ba4f7547.JPG, 190627-091503-corn-
                    nikon_d610-e2cc5642.JPG, 190627-091801-corn-
                    nikon_d610-94fd680b.JPG, 220608-143001-soybeans-
                    HB-25000SBC-37dc1b2c.png, 220608-143001-soybeans-
                    HB-25000SBC-754c6a11.png, 220608-143001-soybeans-
                    HB-25000SBC-8235f317.png, 210625-131202-soybeans-Anafi-
                    fce5a850.JPG, 210625-131205-soybeans-Anafi-8bc119e3.JPG,
                    210625-131303-soybeans-Anafi-614a4aea.JPG,
                    210625-131501-soybeans-Anafi-49b5fd6d.JPG],
              'z': [[0.9999999403953552, 0.9894395470619202, 0.988915205001831,
                    0.9870163798332214, 0.9863036274909973, 0.979924738407135,
                    0.9813047051429749, 0.9824547171592712, 0.9181190729141235,
                    0.9234839081764221, 0.9218310713768005, 0.9175264835357666,
                    0.9

#### t-SNE — 2D

t-SNE preserves local neighborhood structure. Well-separated clusters indicate the model has learned class-discriminative features.

In [6]:
plot_tsne(embeddings, result, output_path=None, dimensions=2)

  t-SNE 2D — fitting 23 samples (perplexity=3, iter=1000)...


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': '%{text}<extra>corn_HB-25000SBC</extra>',
              'marker': {'color': '#1f77b4', 'line': {'color': 'white', 'width': 1}, 'opacity': 0.85, 'size': 10},
              'mode': 'markers',
              'name': 'corn_HB-25000SBC',
              'text': [images/corn_HB-25000SBC/220622-225403-corn-
                       HB-25000SBC-54e94639.png,
                       images/corn_HB-25000SBC/220622-225403-corn-HB-25000SBC-
                       af33e7ca.png, images/corn_HB-25000SBC/220622-225403-corn-
                       HB-25000SBC-fda2adee.png,
                       images/corn_HB-25000SBC/220622-225404-corn-
                       HB-25000SBC-58235c1f.png,
                       images/corn_HB-25000SBC/220622-225405-corn-
                       HB-25000SBC-328a4c61.png,
                       images/corn_HB-25000SBC/220622-225405-corn-
                       HB-25000SBC-68fd040c.png,
                       images/corn_HB-25000SBC/220622-225405-corn-
                       HB-25000SBC-7a2bde6a.png,
                       images/corn_HB-25000SBC/220622-225405-corn-
                       HB-25000SBC-a45a936e.png],
              'type': 'scatter',
              'x': [-40.51651382446289, -39.900962829589844, -37.813785552978516,
                    -40.7496223449707, -39.879180908203125, -44.628475189208984,
                    -42.081363677978516, -36.07853698730469],
              'y': [18.98216438293457, 23.014617919921875, 21.242921829223633,
                    25.275108337402344, 29.697725296020508, 26.81249237060547,
                    30.125965118408203, 29.014789581298828]},
             {'hovertemplate': '%{text}<extra>corn_nikon_d610</extra>',
              'marker': {'color': '#ff7f0e', 'line': {'color': 'white', 'width': 1}, 'opacity': 0.85, 'size': 10},
              'mode': 'markers',
              'name': 'corn_nikon_d610',
              'text': [images/corn_nikon_d610/190627-090900-corn-
                       nikon_d610-bded43cb.JPG,
                       images/corn_nikon_d610/190627-090901-corn-
                       nikon_d610-67679dac.JPG,
                       images/corn_nikon_d610/190627-091105-corn-
                       nikon_d610-8d6b86d0.JPG,
                       images/corn_nikon_d610/190627-091202-corn-
                       nikon_d610-b7725cb0.JPG,
                       images/corn_nikon_d610/190627-090803-corn-
                       nikon_d610-84e374af.JPG,
                       images/corn_nikon_d610/190627-091502-corn-
                       nikon_d610-ba4f7547.JPG,
                       images/corn_nikon_d610/190627-091503-corn-
                       nikon_d610-e2cc5642.JPG,
                       images/corn_nikon_d610/190627-091801-corn-
                       nikon_d610-94fd680b.JPG],
              'type': 'scatter',
              'x': [0.5126943588256836, -6.025384902954102, -6.21523904800415,
                    -4.735989570617676, -0.28283199667930603, -10.454591751098633,
                    -1.0618138313293457, -6.753014087677002],
              'y': [-67.1257553100586, -74.06880950927734, -66.50894165039062,
                    -70.66187286376953, -72.97631072998047, -69.8523941040039,
                    -70.29228973388672, -69.55364990234375]},
             {'hovertemplate': '%{text}<extra>soybean_HB-25000SBC</extra>',
              'marker': {'color': '#2ca02c', 'line': {'color': 'white', 'width': 1}, 'opacity': 0.85, 'size': 10},
              'mode': 'markers',
              'name': 'soybean_HB-25000SBC',
              'text': [images/soybean_HB-25000SBC/220608-143001-soybeans-
                       HB-25000SBC-37dc1b2c.png,
                       images/soybean_HB-25000SBC/220608-143001-soybeans-
                       HB-25000SBC-754c6a11.png,
                       images/soybean_HB-25000SBC/220608-143001-soybeans-
                       HB-25000SBC-8235f317.png],
              'type': 'scatter',
              'x': [5

#### t-SNE — 3D

3D variant — drag to rotate, scroll to zoom.

In [7]:
plot_tsne(embeddings, result, output_path=None, dimensions=3)

  t-SNE 3D — fitting 23 samples (perplexity=3, iter=2000)...


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': '%{text}<extra>corn_HB-25000SBC</extra>',
              'marker': {'color': '#1f77b4', 'line': {'color': 'white', 'width': 1}, 'opacity': 0.85, 'size': 7},
              'mode': 'markers',
              'name': 'corn_HB-25000SBC',
              'text': [images/corn_HB-25000SBC/220622-225403-corn-
                       HB-25000SBC-54e94639.png,
                       images/corn_HB-25000SBC/220622-225403-corn-HB-25000SBC-
                       af33e7ca.png, images/corn_HB-25000SBC/220622-225403-corn-
                       HB-25000SBC-fda2adee.png,
                       images/corn_HB-25000SBC/220622-225404-corn-
                       HB-25000SBC-58235c1f.png,
                       images/corn_HB-25000SBC/220622-225405-corn-
                       HB-25000SBC-328a4c61.png,
                       images/corn_HB-25000SBC/220622-225405-corn-
                       HB-25000SBC-68fd040c.png,
                       images/corn_HB-25000SBC/220622-225405-corn-
                       HB-25000SBC-7a2bde6a.png,
                       images/corn_HB-25000SBC/220622-225405-corn-
                       HB-25000SBC-a45a936e.png],
              'type': 'scatter3d',
              'x': [-152.12672424316406, 66.22169494628906, -62.280364990234375,
                    58.51047897338867, 0.7920692563056946, 56.4085578918457,
                    -44.96760559082031, -112.25345611572266],
              'y': [72.09644317626953, 39.15705490112305, -2.638028144836426,
                    -5.751406192779541, 4.805864334106445, -116.97294616699219,
                    -55.66301345825195, 59.5128059387207],
              'z': [215.0561065673828, -63.4870719909668, 249.24949645996094,
                    137.9437255859375, 34.53290939331055, 171.43869018554688,
                    83.32112884521484, -106.81805419921875]},
             {'hovertemplate': '%{text}<extra>corn_nikon_d610</extra>',
              'marker': {'color': '#ff7f0e', 'line': {'color': 'white', 'width': 1}, 'opacity': 0.85, 'size': 7},
              'mode': 'markers',
              'name': 'corn_nikon_d610',
              'text': [images/corn_nikon_d610/190627-090900-corn-
                       nikon_d610-bded43cb.JPG,
                       images/corn_nikon_d610/190627-090901-corn-
                       nikon_d610-67679dac.JPG,
                       images/corn_nikon_d610/190627-091105-corn-
                       nikon_d610-8d6b86d0.JPG,
                       images/corn_nikon_d610/190627-091202-corn-
                       nikon_d610-b7725cb0.JPG,
                       images/corn_nikon_d610/190627-090803-corn-
                       nikon_d610-84e374af.JPG,
                       images/corn_nikon_d610/190627-091502-corn-
                       nikon_d610-ba4f7547.JPG,
                       images/corn_nikon_d610/190627-091503-corn-
                       nikon_d610-e2cc5642.JPG,
                       images/corn_nikon_d610/190627-091801-corn-
                       nikon_d610-94fd680b.JPG],
              'type': 'scatter3d',
              'x': [17.048702239990234, 38.85808563232422, 165.03892517089844,
                    -14.31806755065918, 18.485570907592773, -135.5693359375,
                    65.5746841430664, -97.22266387939453],
              'y': [-220.50430297851562, 217.70608520507812, 107.78890991210938,
                    -90.82244873046875, 171.91586303710938, -129.8266143798828,
                    -161.73873901367188, -119.34650421142578],
              'z': [-16.155475616455078, 178.8775177001953, 183.023681640625,
                    -109.09793090820312, -182.60386657714844, -197.2759552001953,
                    -102.24263000488281, -93.94332122802734]},
             {'hovertemplate': '%{text}<extra>soybean_HB-25000SBC</extra>',
              'marker': {'color': '#2ca02c', 'line': {'color': 'white', 'width': 1}, 'opacity': 0.85, 'size': 7},
              'mode': 'markers',
              'name': 'soybean_HB-25000S

#### LLE — 3D

Locally Linear Embedding preserves local geometry rather than global distances, complementing the t-SNE view. Drag to rotate.

In [8]:
plot_lle(embeddings, result, output_path=None)

  LLE 3D — fitting 23 samples (n_neighbors=3)...


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': '%{text}<extra>corn_HB-25000SBC</extra>',
              'marker': {'color': '#1f77b4', 'line': {'color': 'white', 'width': 1}, 'opacity': 0.85, 'size': 7},
              'mode': 'markers',
              'name': 'corn_HB-25000SBC',
              'text': [images/corn_HB-25000SBC/220622-225403-corn-
                       HB-25000SBC-54e94639.png,
                       images/corn_HB-25000SBC/220622-225403-corn-HB-25000SBC-
                       af33e7ca.png, images/corn_HB-25000SBC/220622-225403-corn-
                       HB-25000SBC-fda2adee.png,
                       images/corn_HB-25000SBC/220622-225404-corn-
                       HB-25000SBC-58235c1f.png,
                       images/corn_HB-25000SBC/220622-225405-corn-
                       HB-25000SBC-328a4c61.png,
                       images/corn_HB-25000SBC/220622-225405-corn-
                       HB-25000SBC-68fd040c.png,
                       images/corn_HB-25000SBC/220622-225405-corn-
                       HB-25000SBC-7a2bde6a.png,
                       images/corn_HB-25000SBC/220622-225405-corn-
                       HB-25000SBC-a45a936e.png],
              'type': 'scatter3d',
              'x': [-0.3535533905932737, -0.3535533905932735,
                    -0.35355339059327345, -0.35355339059327384,
                    -0.3535533905932736, -0.3535533905932741, -0.35355339059327373,
                    -0.3535533905932737],
              'y': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
              'z': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]},
             {'hovertemplate': '%{text}<extra>corn_nikon_d610</extra>',
              'marker': {'color': '#ff7f0e', 'line': {'color': 'white', 'width': 1}, 'opacity': 0.85, 'size': 7},
              'mode': 'markers',
              'name': 'corn_nikon_d610',
              'text': [images/corn_nikon_d610/190627-090900-corn-
                       nikon_d610-bded43cb.JPG,
                       images/corn_nikon_d610/190627-090901-corn-
                       nikon_d610-67679dac.JPG,
                       images/corn_nikon_d610/190627-091105-corn-
                       nikon_d610-8d6b86d0.JPG,
                       images/corn_nikon_d610/190627-091202-corn-
                       nikon_d610-b7725cb0.JPG,
                       images/corn_nikon_d610/190627-090803-corn-
                       nikon_d610-84e374af.JPG,
                       images/corn_nikon_d610/190627-091502-corn-
                       nikon_d610-ba4f7547.JPG,
                       images/corn_nikon_d610/190627-091503-corn-
                       nikon_d610-e2cc5642.JPG,
                       images/corn_nikon_d610/190627-091801-corn-
                       nikon_d610-94fd680b.JPG],
              'type': 'scatter3d',
              'x': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
              'y': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
              'z': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]},
             {'hovertemplate': '%{text}<extra>soybean_HB-25000SBC</extra>',
              'marker': {'color': '#2ca02c', 'line': {'color': 'white', 'width': 1}, 'opacity': 0.85, 'size': 7},
              'mode': 'markers',
              'name': 'soybean_HB-25000SBC',
              'text': [images/soybean_HB-25000SBC/220608-143001-soybeans-
                       HB-25000SBC-37dc1b2c.png,
                       images/soybean_HB-25000SBC/220608-143001-soybeans-
                       HB-25000SBC-754c6a11.png,
                       images/soybean_HB-25000SBC/220608-143001-soybeans-
                       HB-25000SBC-8235f317.png],
              'type': 'scatter3d',
              'x': [0.0, 0.0, 0.0],
              'y': [-0.3779644730092353, -0.37796447300923575,
                    -0.37796447300923486],
              'z': [0.42825947740788595, 0.47112744882516105, 0.4085736998439049]},
             {'hovertemplate': '%{text}<extra>soybean_anafi</extra>',
              'marker': {'color': '#d62728', 'line':

---

## Sparse Scenario

Same image paths and class structure as the tight scenario, but class prototypes are close together (~60° apart) and per-sample noise is large. Compare KPIs and plots directly against the tight scenario above to see how embedding quality degrades.

### Load Data

In [9]:
sparse_path = Path("emb_sparse.json")
with open(sparse_path) as f:
    sparse_payload = json.load(f)

sparse_embeddings: dict[str, list[float]] = sparse_payload["embeddings"]
print(f"Loaded {len(sparse_embeddings)} sparse embeddings  (dim={len(next(iter(sparse_embeddings.values())))})")

Loaded 23 sparse embeddings  (dim=32)


### Run Evaluation

In [10]:
sparse_result = run_evaluation(
    image_embeddings=sparse_embeddings,
    k_values=[5, 10],
    dataset_root="images",
    sample_pairs=None,
)

print_result(sparse_result)

Confusion matrix: 100%|██████████| 8/8 [00:00<00:00, 101.71step/s]        

n_items      : 23
embedding_dim: 32
classes      : ['corn', 'soybean']
k_values     : [5, 10]

── global_metrics ──────────────────────────────────────────────────────
  pairwise cosine    : mean=0.2989  std=0.2826  (p05=-0.1287  p50=0.3227  p95=0.7003)
  centroid cosine    : mean=0.5739  std=0.1942  norm=0.5739
  intra/inter gap    : 0.4894  (intra=0.5155  inter=0.0262)
  effective_rank     : 6.25  (ratio=0.1952  dim=32)
  uniformity         : -2.2539

  hubness@5         : mean=5.0000  std=2.5022  p95=8.0000
  hubness@10        : mean=10.0000  std=3.9009  p95=16.0000
  knn_radius@5         : mean=0.5065  std=0.1601  p05=0.2282  p95=0.6873
  knn_radius@10        : mean=0.3718  std=0.1806  p05=0.0325  p95=0.5393
  mean_top_k_sim@5         : mean=0.6078  std=0.1042  p05=0.4109  p95=0.7241
  mean_top_k_sim@10        : mean=0.5171  std=0.1358  p05=0.3094  p95=0.6599
  outlier_score@5         : mean=0.3922  std=0.1042  p95=0.5891
  outlier_score@10        : mean=0.4829  std=0.1358  p95=0.6

### Visualizations

All plots are interactive — hover for details, click legend entries to toggle classes, and drag to rotate 3D views.

#### KNN Confusion Matrix

Rows = true class, columns = neighbor class, values = fraction of k-NN neighbors belonging to each class. The diagonal equals mean KNN purity — higher is better.

In [11]:
plot_knn_confusion(sparse_result, output_path=None)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'colorbar': {'thickness': 18, 'title': {'text': 'KNN@10<br>neighbor<br>fraction'}},
              'colorscale': [[0.0, 'rgb(165,0,38)'], [0.1, 'rgb(215,48,39)'],
                             [0.2, 'rgb(244,109,67)'], [0.3, 'rgb(253,174,97)'],
                             [0.4, 'rgb(254,224,139)'], [0.5, 'rgb(255,255,191)'],
                             [0.6, 'rgb(217,239,139)'], [0.7, 'rgb(166,217,106)'],
                             [0.8, 'rgb(102,189,99)'], [0.9, 'rgb(26,152,80)'],
                             [1.0, 'rgb(0,104,55)']],
              'hovertemplate': ('True class: <b>%{y}</b><br>Nei' ... 'ction: %{z:.4f}<extra></extra>'),
              'text': [['0.994', '0.006'], ['0.443', '0.557']],
              'textfont': {'size': 13},
              'texttemplate': '%{text}',
              'type': 'heatmap',
              'x': [corn, soybean],
              'y': [corn, soybean],
              'z': [[0.99375, 0.00625], [0.44285714285714284, 0.5571428571428572]],
              'zmax': 1.0,
              'zmin': 0.0}],
    'layout': {'height': 530,
               'margin': {'b': 100, 'l': 120, 'r': 60, 't': 70},
               'paper_bgcolor': 'white',
               'plot_bgcolor': 'white',
               'template': '...',
               'title': {'font': {'size': 17}, 'text': 'KNN Confusion Matrix  (k=10)'},
               'width': 560,
               'xaxis': {'side': 'bottom', 'tickfont': {'size': 12}, 'title': {'text': 'Neighbor class'}},
               'yaxis': {'autorange': 'reversed', 'tickfont': {'size': 12}, 'title': {'text': 'True class'}}}
})

#### Pairwise Cosine Similarity

Full N×N cosine similarity matrix sorted by class. Within-class blocks sit on the diagonal — tighter, brighter blocks indicate a more discriminative embedding space.

In [12]:
plot_cosine_similarity(sparse_embeddings, sparse_result, output_path=None)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'colorbar': {'thickness': 18, 'title': {'text': 'Cosine<br>similarity'}},
              'colorscale': [[0.0, 'rgb(5,48,97)'], [0.1, 'rgb(33,102,172)'],
                             [0.2, 'rgb(67,147,195)'], [0.3, 'rgb(146,197,222)'],
                             [0.4, 'rgb(209,229,240)'], [0.5, 'rgb(247,247,247)'],
                             [0.6, 'rgb(253,219,199)'], [0.7, 'rgb(244,165,130)'],
                             [0.8, 'rgb(214,96,77)'], [0.9, 'rgb(178,24,43)'],
                             [1.0, 'rgb(103,0,31)']],
              'hovertemplate': 'Row: %{y}<br>Col: %{x}<br>Cosine similarity: %{z:.4f}<extra></extra>',
              'type': 'heatmap',
              'x': [220622-225403-corn-HB-25000SBC-54e94639.png,
                    220622-225403-corn-HB-25000SBC-af33e7ca.png,
                    220622-225403-corn-HB-25000SBC-fda2adee.png,
                    220622-225404-corn-HB-25000SBC-58235c1f.png,
                    220622-225405-corn-HB-25000SBC-328a4c61.png,
                    220622-225405-corn-HB-25000SBC-68fd040c.png,
                    220622-225405-corn-HB-25000SBC-7a2bde6a.png,
                    220622-225405-corn-HB-25000SBC-a45a936e.png,
                    190627-090900-corn-nikon_d610-bded43cb.JPG, 190627-090901-corn-
                    nikon_d610-67679dac.JPG, 190627-091105-corn-
                    nikon_d610-8d6b86d0.JPG, 190627-091202-corn-
                    nikon_d610-b7725cb0.JPG, 190627-090803-corn-
                    nikon_d610-84e374af.JPG, 190627-091502-corn-
                    nikon_d610-ba4f7547.JPG, 190627-091503-corn-
                    nikon_d610-e2cc5642.JPG, 190627-091801-corn-
                    nikon_d610-94fd680b.JPG, 220608-143001-soybeans-
                    HB-25000SBC-37dc1b2c.png, 220608-143001-soybeans-
                    HB-25000SBC-754c6a11.png, 220608-143001-soybeans-
                    HB-25000SBC-8235f317.png, 210625-131202-soybeans-Anafi-
                    fce5a850.JPG, 210625-131205-soybeans-Anafi-8bc119e3.JPG,
                    210625-131303-soybeans-Anafi-614a4aea.JPG,
                    210625-131501-soybeans-Anafi-49b5fd6d.JPG],
              'y': [220622-225403-corn-HB-25000SBC-54e94639.png,
                    220622-225403-corn-HB-25000SBC-af33e7ca.png,
                    220622-225403-corn-HB-25000SBC-fda2adee.png,
                    220622-225404-corn-HB-25000SBC-58235c1f.png,
                    220622-225405-corn-HB-25000SBC-328a4c61.png,
                    220622-225405-corn-HB-25000SBC-68fd040c.png,
                    220622-225405-corn-HB-25000SBC-7a2bde6a.png,
                    220622-225405-corn-HB-25000SBC-a45a936e.png,
                    190627-090900-corn-nikon_d610-bded43cb.JPG, 190627-090901-corn-
                    nikon_d610-67679dac.JPG, 190627-091105-corn-
                    nikon_d610-8d6b86d0.JPG, 190627-091202-corn-
                    nikon_d610-b7725cb0.JPG, 190627-090803-corn-
                    nikon_d610-84e374af.JPG, 190627-091502-corn-
                    nikon_d610-ba4f7547.JPG, 190627-091503-corn-
                    nikon_d610-e2cc5642.JPG, 190627-091801-corn-
                    nikon_d610-94fd680b.JPG, 220608-143001-soybeans-
                    HB-25000SBC-37dc1b2c.png, 220608-143001-soybeans-
                    HB-25000SBC-754c6a11.png, 220608-143001-soybeans-
                    HB-25000SBC-8235f317.png, 210625-131202-soybeans-Anafi-
                    fce5a850.JPG, 210625-131205-soybeans-Anafi-8bc119e3.JPG,
                    210625-131303-soybeans-Anafi-614a4aea.JPG,
                    210625-131501-soybeans-Anafi-49b5fd6d.JPG],
              'z': [[0.9999998807907104, 0.6308585405349731, 0.5869923233985901,
                    0.4986586272716522, 0.501905083656311, 0.32273200154304504,
                    0.35237497091293335, 0.4504513442516327, 0.20318207144737244,
                    0.4080139398574829, 0.35081738233566284, 0.2999541461467743,
                  

#### t-SNE — 2D

t-SNE preserves local neighborhood structure. Well-separated clusters indicate the model has learned class-discriminative features.

In [13]:
plot_tsne(sparse_embeddings, sparse_result, output_path=None, dimensions=2)

  t-SNE 2D — fitting 23 samples (perplexity=3, iter=1000)...


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': '%{text}<extra>corn_HB-25000SBC</extra>',
              'marker': {'color': '#1f77b4', 'line': {'color': 'white', 'width': 1}, 'opacity': 0.85, 'size': 10},
              'mode': 'markers',
              'name': 'corn_HB-25000SBC',
              'text': [images/corn_HB-25000SBC/220622-225403-corn-
                       HB-25000SBC-54e94639.png,
                       images/corn_HB-25000SBC/220622-225403-corn-HB-25000SBC-
                       af33e7ca.png, images/corn_HB-25000SBC/220622-225403-corn-
                       HB-25000SBC-fda2adee.png,
                       images/corn_HB-25000SBC/220622-225404-corn-
                       HB-25000SBC-58235c1f.png,
                       images/corn_HB-25000SBC/220622-225405-corn-
                       HB-25000SBC-328a4c61.png,
                       images/corn_HB-25000SBC/220622-225405-corn-
                       HB-25000SBC-68fd040c.png,
                       images/corn_HB-25000SBC/220622-225405-corn-
                       HB-25000SBC-7a2bde6a.png,
                       images/corn_HB-25000SBC/220622-225405-corn-
                       HB-25000SBC-a45a936e.png],
              'type': 'scatter',
              'x': [22.272294998168945, 13.521811485290527, 11.425545692443848,
                    10.300097465515137, 17.06630516052246, 2.0326969623565674,
                    10.948486328125, 43.10585021972656],
              'y': [86.00914001464844, 79.8515625, 87.71582794189453,
                    74.25533294677734, 61.706912994384766, 64.98566436767578,
                    60.26789474487305, 37.66725158691406]},
             {'hovertemplate': '%{text}<extra>corn_nikon_d610</extra>',
              'marker': {'color': '#ff7f0e', 'line': {'color': 'white', 'width': 1}, 'opacity': 0.85, 'size': 10},
              'mode': 'markers',
              'name': 'corn_nikon_d610',
              'text': [images/corn_nikon_d610/190627-090900-corn-
                       nikon_d610-bded43cb.JPG,
                       images/corn_nikon_d610/190627-090901-corn-
                       nikon_d610-67679dac.JPG,
                       images/corn_nikon_d610/190627-091105-corn-
                       nikon_d610-8d6b86d0.JPG,
                       images/corn_nikon_d610/190627-091202-corn-
                       nikon_d610-b7725cb0.JPG,
                       images/corn_nikon_d610/190627-090803-corn-
                       nikon_d610-84e374af.JPG,
                       images/corn_nikon_d610/190627-091502-corn-
                       nikon_d610-ba4f7547.JPG,
                       images/corn_nikon_d610/190627-091503-corn-
                       nikon_d610-e2cc5642.JPG,
                       images/corn_nikon_d610/190627-091801-corn-
                       nikon_d610-94fd680b.JPG],
              'type': 'scatter',
              'x': [45.01708221435547, 68.13316345214844, 61.2188606262207,
                    60.199153900146484, 59.23170852661133, 75.00116729736328,
                    51.127655029296875, 66.817626953125],
              'y': [20.89293670654297, 6.650092601776123, 14.334636688232422,
                    22.451274871826172, 35.16815948486328, 25.27887725830078,
                    29.971174240112305, 20.38121795654297]},
             {'hovertemplate': '%{text}<extra>soybean_HB-25000SBC</extra>',
              'marker': {'color': '#2ca02c', 'line': {'color': 'white', 'width': 1}, 'opacity': 0.85, 'size': 10},
              'mode': 'markers',
              'name': 'soybean_HB-25000SBC',
              'text': [images/soybean_HB-25000SBC/220608-143001-soybeans-
                       HB-25000SBC-37dc1b2c.png,
                       images/soybean_HB-25000SBC/220608-143001-soybeans-
                       HB-25000SBC-754c6a11.png,
                       images/soybean_HB-25000SBC/220608-143001-soybeans-
                       HB-25000SBC-8235f317.png],
              'type': 'scatter',
              'x': [-49.72450637817383, -60.0607299804687

#### t-SNE — 3D

3D variant — drag to rotate, scroll to zoom.

In [14]:
plot_tsne(sparse_embeddings, sparse_result, output_path=None, dimensions=3)

  t-SNE 3D — fitting 23 samples (perplexity=3, iter=2000)...


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': '%{text}<extra>corn_HB-25000SBC</extra>',
              'marker': {'color': '#1f77b4', 'line': {'color': 'white', 'width': 1}, 'opacity': 0.85, 'size': 7},
              'mode': 'markers',
              'name': 'corn_HB-25000SBC',
              'text': [images/corn_HB-25000SBC/220622-225403-corn-
                       HB-25000SBC-54e94639.png,
                       images/corn_HB-25000SBC/220622-225403-corn-HB-25000SBC-
                       af33e7ca.png, images/corn_HB-25000SBC/220622-225403-corn-
                       HB-25000SBC-fda2adee.png,
                       images/corn_HB-25000SBC/220622-225404-corn-
                       HB-25000SBC-58235c1f.png,
                       images/corn_HB-25000SBC/220622-225405-corn-
                       HB-25000SBC-328a4c61.png,
                       images/corn_HB-25000SBC/220622-225405-corn-
                       HB-25000SBC-68fd040c.png,
                       images/corn_HB-25000SBC/220622-225405-corn-
                       HB-25000SBC-7a2bde6a.png,
                       images/corn_HB-25000SBC/220622-225405-corn-
                       HB-25000SBC-a45a936e.png],
              'type': 'scatter3d',
              'x': [279.0123291015625, -20.568958282470703, -287.2744445800781,
                    140.5340118408203, -169.77957153320312, 157.61968994140625,
                    -199.86572265625, -28.42926788330078],
              'y': [103.26578521728516, -177.7697296142578, -141.85414123535156,
                    -91.43517303466797, 19.749788284301758, -202.61524963378906,
                    -56.78934097290039, 16.747859954833984],
              'z': [-309.6524963378906, 79.07393646240234, 183.7709197998047,
                    192.0282745361328, -190.6502227783203, 111.65618133544922,
                    -112.06632995605469, -75.46227264404297]},
             {'hovertemplate': '%{text}<extra>corn_nikon_d610</extra>',
              'marker': {'color': '#ff7f0e', 'line': {'color': 'white', 'width': 1}, 'opacity': 0.85, 'size': 7},
              'mode': 'markers',
              'name': 'corn_nikon_d610',
              'text': [images/corn_nikon_d610/190627-090900-corn-
                       nikon_d610-bded43cb.JPG,
                       images/corn_nikon_d610/190627-090901-corn-
                       nikon_d610-67679dac.JPG,
                       images/corn_nikon_d610/190627-091105-corn-
                       nikon_d610-8d6b86d0.JPG,
                       images/corn_nikon_d610/190627-091202-corn-
                       nikon_d610-b7725cb0.JPG,
                       images/corn_nikon_d610/190627-090803-corn-
                       nikon_d610-84e374af.JPG,
                       images/corn_nikon_d610/190627-091502-corn-
                       nikon_d610-ba4f7547.JPG,
                       images/corn_nikon_d610/190627-091503-corn-
                       nikon_d610-e2cc5642.JPG,
                       images/corn_nikon_d610/190627-091801-corn-
                       nikon_d610-94fd680b.JPG],
              'type': 'scatter3d',
              'x': [189.23291015625, 43.59453201293945, -135.7079315185547,
                    47.08943557739258, 46.75994873046875, 36.88278579711914,
                    53.36159133911133, 230.96337890625],
              'y': [216.49327087402344, -304.2447204589844, -179.04861450195312,
                    193.4516143798828, 49.364418029785156, 160.38262939453125,
                    103.52700805664062, 68.55272674560547],
              'z': [9.856649398803711, -117.61229705810547, -231.85850524902344,
                    67.54092407226562, 110.14229583740234, 261.1436767578125,
                    -18.876407623291016, 57.58906173706055]},
             {'hovertemplate': '%{text}<extra>soybean_HB-25000SBC</extra>',
              'marker': {'color': '#2ca02c', 'line': {'color': 'white', 'width': 1}, 'opacity': 0.85, 'size': 7},
              'mode': 'markers',
              'name': 'soybean_HB-25000SBC',

#### LLE — 3D

Locally Linear Embedding preserves local geometry rather than global distances, complementing the t-SNE view. Drag to rotate.

In [15]:
plot_lle(sparse_embeddings, sparse_result, output_path=None)

  LLE 3D — fitting 23 samples (n_neighbors=3)...


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': '%{text}<extra>corn_HB-25000SBC</extra>',
              'marker': {'color': '#1f77b4', 'line': {'color': 'white', 'width': 1}, 'opacity': 0.85, 'size': 7},
              'mode': 'markers',
              'name': 'corn_HB-25000SBC',
              'text': [images/corn_HB-25000SBC/220622-225403-corn-
                       HB-25000SBC-54e94639.png,
                       images/corn_HB-25000SBC/220622-225403-corn-HB-25000SBC-
                       af33e7ca.png, images/corn_HB-25000SBC/220622-225403-corn-
                       HB-25000SBC-fda2adee.png,
                       images/corn_HB-25000SBC/220622-225404-corn-
                       HB-25000SBC-58235c1f.png,
                       images/corn_HB-25000SBC/220622-225405-corn-
                       HB-25000SBC-328a4c61.png,
                       images/corn_HB-25000SBC/220622-225405-corn-
                       HB-25000SBC-68fd040c.png,
                       images/corn_HB-25000SBC/220622-225405-corn-
                       HB-25000SBC-7a2bde6a.png,
                       images/corn_HB-25000SBC/220622-225405-corn-
                       HB-25000SBC-a45a936e.png],
              'type': 'scatter3d',
              'x': [-0.002741135219513562, -0.0027411352195133424,
                    -0.0027411352195135697, -0.002741135219513653,
                    -0.002741135219513707, -0.0027411352195132925,
                    -0.0027411352195136998, -0.0027411352195129287],
              'y': [0.27111550995186173, 0.21170266214696243, 0.2797018692019576,
                    0.30761147173021824, 0.3191661242180313, 0.12639714449329786,
                    0.3102209906950881, 0.017059164329616662],
              'z': [-0.05496897502767295, -0.009940980012597925,
                    -0.04069561428823322, 0.08736944778868372, 0.1600838372726973,
                    0.12662269254059275, 0.1999374646070492, 0.05422777342161977]},
             {'hovertemplate': '%{text}<extra>corn_nikon_d610</extra>',
              'marker': {'color': '#ff7f0e', 'line': {'color': 'white', 'width': 1}, 'opacity': 0.85, 'size': 7},
              'mode': 'markers',
              'name': 'corn_nikon_d610',
              'text': [images/corn_nikon_d610/190627-090900-corn-
                       nikon_d610-bded43cb.JPG,
                       images/corn_nikon_d610/190627-090901-corn-
                       nikon_d610-67679dac.JPG,
                       images/corn_nikon_d610/190627-091105-corn-
                       nikon_d610-8d6b86d0.JPG,
                       images/corn_nikon_d610/190627-091202-corn-
                       nikon_d610-b7725cb0.JPG,
                       images/corn_nikon_d610/190627-090803-corn-
                       nikon_d610-84e374af.JPG,
                       images/corn_nikon_d610/190627-091502-corn-
                       nikon_d610-ba4f7547.JPG,
                       images/corn_nikon_d610/190627-091503-corn-
                       nikon_d610-e2cc5642.JPG,
                       images/corn_nikon_d610/190627-091801-corn-
                       nikon_d610-94fd680b.JPG],
              'type': 'scatter3d',
              'x': [-0.0027411352195123714, -0.0027411352195125084,
                    -0.0027411352195125436, -0.0027411352195124425,
                    -0.002741135219512175, -0.0027411352195120886,
                    -0.002741135219512353, -0.002741135219512297],
              'y': [-0.2470254544130036, -0.2330230315331629,
                    -0.22570508697632746, -0.23928246229980096,
                    -0.24911196452320622, -0.24919716763590782,
                    -0.2481636978922463, -0.24101681066102318],
              'z': [-0.009135260965190193, 0.0007018686613822786,
                    -0.012291735666114906, -0.0063817638226528126,
                    -0.005024424011795791, -0.010143812744854689,
                    -0.006407169264324013, -0.011315863291849793]},
             {'hovertemplate': '%{text}<extra>soybean_HB-250